## Clase 2 - Exploracion y preparacion de datos
ENIGH 2024, hogares de Chiapas. 8 ejercicios cortos que conectan teoria (tipos, descriptiva, distancias, calidad, transformacion) con un dataset real.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mineria_utils import (
    NAVY, TERRA, SAND, INK, PLOTLY_TEMPLATE, PLOTLY_FONT,
    clasificar_atributos, detectar_outliers_iqr, chi_square,
    aplicar_normalizaciones, distancia_matriz, imputar_por_grupo,
)

CSV_PATH = "./data/enigh_chiapas_2024.csv"
if not Path(CSV_PATH).exists():
    CSV_PATH = "../data/enigh_chiapas_2024.csv"
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
df.head(3)

### 1. Clasificar atributos por tipo
El dtype de pandas no es el tipo conceptual. `folioviv` y `ubica_geo` son `int64` pero son identificadores nominales; `tam_loc` y `est_socio` son enteros pero ordinales.

In [ ]:
HINTS = {
    "folioviv": "nominal",
    "ubica_geo": "nominal",
    "clase_hog": "nominal",
    "tam_loc": "ordinal",
    "est_socio": "ordinal",
    "educa_jefe": "ordinal",
    "sexo_jefe": "binario",
}
clasificar_atributos(df, hints=HINTS)

### 2. Descriptiva y la trampa de la media
Regla de Han para diagnosticar asimetria: `media - moda` ~ `3 * (media - mediana)`. En distribuciones de ingreso esperamos sesgo positivo fuerte.

In [ ]:
num_cols = ["edad_jefe", "tot_integ", "ing_cor", "gasto_mon", "alimentos", "salud", "transporte"]
df[num_cols].describe().T.round(2)

In [ ]:
ic = df["ing_cor"].dropna()
print(f"Media:    {ic.mean():>12,.0f} MXN")
print(f"Mediana:  {ic.median():>12,.0f} MXN")
print(f"Moda:     {ic.mode().iloc[0]:>12,.0f} MXN")
print(f"Diferencia media vs mediana: {(ic.mean() - ic.median()) / ic.median():.1%}")

In [ ]:
fig = px.histogram(df, x="ing_cor", nbins=60, template=PLOTLY_TEMPLATE,
                   color_discrete_sequence=[NAVY])
fig.add_vline(x=ic.mean(),   line_color=TERRA, line_width=2, annotation_text="media")
fig.add_vline(x=ic.median(), line_color=NAVY,  line_width=2, line_dash="dash", annotation_text="mediana")
fig.update_layout(title="Ingreso corriente trimestral - hogares de Chiapas",
                  font=PLOTLY_FONT, width=900, height=450)
fig.show()

### 3. Outliers con la regla 1.5 IQR
Marcamos como outlier todo valor fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`. Detectar no significa borrar.

In [ ]:
mask = detectar_outliers_iqr(df["ing_cor"])
n_out = mask.sum()
print(f"Outliers en ing_cor: {n_out:,} de {len(df):,} hogares ({n_out/len(df):.1%})")

fig = px.box(df, y="ing_cor", points="outliers", template=PLOTLY_TEMPLATE,
             color_discrete_sequence=[NAVY])
fig.update_layout(title="ing_cor - distribucion y outliers (regla 1.5 IQR)",
                  font=PLOTLY_FONT, width=600, height=550)
fig.show()

### 4. Correlacion numerica y chi-cuadrado nominal
Pearson mide asociacion lineal entre numericas. Para nominales necesitamos chi-cuadrado de independencia sobre la tabla de contingencia.

In [ ]:
corr_cols = ["edad_jefe", "tot_integ", "ing_cor", "ingtrab", "gasto_mon", "alimentos", "salud", "transporte"]
corr = df[corr_cols].corr().round(2)
fig = px.imshow(corr, text_auto=True,
                color_continuous_scale=[(0, SAND), (0.5, "white"), (1, NAVY)],
                zmin=-1, zmax=1, template=PLOTLY_TEMPLATE)
fig.update_layout(title="Correlacion Pearson", font=PLOTLY_FONT, width=700, height=600)
fig.show()

In [ ]:
res1 = chi_square(df, "sexo_jefe", "clase_hog")
res2 = chi_square(df, "tam_loc",   "clase_hog")
print(f"sexo_jefe x clase_hog:  chi2 = {res1['chi2']:.2f},  p = {res1['p_value']:.4g},  dof = {res1['dof']}")
print(f"tam_loc   x clase_hog:  chi2 = {res2['chi2']:.2f},  p = {res2['p_value']:.4g},  dof = {res2['dof']}")
print("\nTabla de contingencia sexo_jefe x clase_hog:")
display(res1["observed"])

### 5. Tres estrategias de imputacion
El CSV ya viene sin NaN, asi que introducimos un 10% artificial sobre `ing_cor` para comparar el efecto de cada estrategia sobre la varianza.

In [ ]:
rng = np.random.default_rng(42)
df_miss = df.copy()
mask_miss = rng.random(len(df_miss)) < 0.10
df_miss.loc[mask_miss, "ing_cor"] = np.nan
print(f"Missing introducidos: {mask_miss.sum():,} ({mask_miss.mean():.1%})")

In [ ]:
s_dropna = df_miss["ing_cor"].dropna()
s_median = df_miss["ing_cor"].fillna(df_miss["ing_cor"].median())
s_grupo  = imputar_por_grupo(df_miss, target="ing_cor", group="est_socio", estrategia="median")

fig = make_subplots(rows=1, cols=3, subplot_titles=("Drop NA", "Mediana global", "Mediana por est_socio"))
for i, (s, color) in enumerate(zip([s_dropna, s_median, s_grupo], [NAVY, TERRA, NAVY])):
    fig.add_trace(go.Histogram(x=s, nbinsx=50, marker_color=color, showlegend=False), row=1, col=i+1)
fig.update_layout(template=PLOTLY_TEMPLATE, font=PLOTLY_FONT, width=1200, height=400,
                  title="Comparacion de estrategias de imputacion")
fig.show()

print(f"Std original:           {df['ing_cor'].std():>12,.0f}")
print(f"Std tras dropna:        {s_dropna.std():>12,.0f}")
print(f"Std tras mediana glob:  {s_median.std():>12,.0f}  <- varianza colapsada")
print(f"Std tras mediana grupo: {s_grupo.std():>12,.0f}")

### 6. Tres normalizaciones
Min-max lleva a `[0, 1]`, z-score centra y escala por desviacion, decimal scaling divide por la potencia de 10 mas cercana al maximo.

In [ ]:
norms = aplicar_normalizaciones(df["ing_cor"].dropna())
display(norms.describe().round(3))

fig = make_subplots(rows=1, cols=3, subplot_titles=("Min-Max [0,1]", "Z-score", "Decimal scaling"))
for i, col in enumerate(["minmax", "zscore", "decimal"]):
    fig.add_trace(go.Histogram(x=norms[col], nbinsx=50, marker_color=NAVY, showlegend=False), row=1, col=i+1)
fig.update_layout(template=PLOTLY_TEMPLATE, font=PLOTLY_FONT, width=1200, height=400,
                  title="Tres normalizaciones de ing_cor")
fig.show()

### 7. Distancias entre hogares
Manhattan suma diferencias absolutas, Euclidean penaliza mas las diferencias grandes. La eleccion cambia el ranking de vecinos.

In [ ]:
sample = df[corr_cols].dropna().sample(5, random_state=7).reset_index(drop=True)
sample_z = (sample - sample.mean()) / sample.std()

D_euc = distancia_matriz(sample_z, metrica="euclidean")
D_man = distancia_matriz(sample_z, metrica="manhattan")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Euclidean", "Manhattan"))
fig.add_trace(go.Heatmap(z=D_euc, colorscale=[(0, SAND), (1, NAVY)], showscale=False,
                          text=np.round(D_euc, 2), texttemplate="%{text}"), row=1, col=1)
fig.add_trace(go.Heatmap(z=D_man, colorscale=[(0, SAND), (1, TERRA)], showscale=False,
                          text=np.round(D_man, 2), texttemplate="%{text}"), row=1, col=2)
fig.update_layout(template=PLOTLY_TEMPLATE, font=PLOTLY_FONT, width=1000, height=450,
                  title="Distancias pareja a pareja entre 5 hogares")
fig.show()

### 8. PCA 2D y visualizacion
PCA es una proyeccion lineal que maximiza varianza. `explained_variance_ratio_` nos dice cuanta informacion conservan los primeros componentes.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

cols_pca = corr_cols
df_pca = df[cols_pca + ["est_socio"]].dropna()
X = StandardScaler().fit_transform(df_pca[cols_pca])
pca = PCA(n_components=2).fit(X)
pcs = pca.transform(X)

print(f"Varianza explicada por PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"Varianza explicada por PC2: {pca.explained_variance_ratio_[1]:.1%}")
print(f"Acumulada (2 PC):           {pca.explained_variance_ratio_.sum():.1%}")

df_plot = df_pca.assign(PC1=pcs[:, 0], PC2=pcs[:, 1])
fig = px.scatter(df_plot, x="PC1", y="PC2", color="est_socio",
                 color_continuous_scale=[(0, TERRA), (1, NAVY)],
                 template=PLOTLY_TEMPLATE, opacity=0.6)
fig.update_layout(title="PCA 2D - coloreado por estrato socioeconomico",
                  font=PLOTLY_FONT, width=900, height=550)
fig.show()

### Cierre - mapa de decision aplicado
- Tipo conceptual primero, dtype despues: `folioviv` numerica almacenada no implica numerica analizable.
- Si la distribucion es asimetrica (media >> mediana), reportar mediana y discutir cola, no recortar a ciegas.
- Outliers se investigan antes de borrarse; el 5-10% superior de `ing_cor` es senal, no ruido.
- Asociacion depende del tipo: Pearson para numerico-numerico, chi-cuadrado para nominal-nominal.
- Imputar global colapsa varianza; imputar por grupo preserva estructura. Normalizar no arregla sesgo.